# 03 — Model Comparison

Training logs, hyperparameter tuning experiments, performance metrics, and comparative visualizations for ARIMA, LSTM, and XGBoost.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_processed_data
from src.evaluation import compare_models, compute_metrics, plot_metric_comparison, plot_predictions
from src.models.arima_model import ARIMAModel
from src.models.lstm_model import LSTMModel, create_sequences
from src.models.xgboost_model import XGBoostModel
from src.preprocessor import chronological_split, SeriesScaler
from src.training import grid_search, log_experiment
from src.utils import load_config, save_fig, set_seed

cfg = load_config('../config.yaml')
set_seed(cfg['seed'])
data_cfg, model_cfg, paths_cfg = cfg['data'], cfg['models'], cfg['paths']
target = data_cfg['target_column']

features_df = load_processed_data('../' + data_cfg['processed_path'])
train_df, val_df, test_df = chronological_split(features_df, data_cfg['test_size'], data_cfg['validation_size'])
feature_cols = [c for c in features_df.columns if c != target]

## 1. Train ARIMA

In [ ]:
arima_cfg = model_cfg['arima']
arima = ARIMAModel(tuple(arima_cfg['order']), tuple(arima_cfg['seasonal_order']))
arima.fit(train_df[target])
arima_preds = arima.predict(steps=len(test_df))
arima_metrics = compute_metrics(test_df[target].values, arima_preds)
arima_metrics

## 2. Train LSTM

In [ ]:
lstm_cfg = model_cfg['lstm']
scaler = SeriesScaler(cfg['preprocessing']['scaling_method'])
train_scaled = scaler.fit_transform(train_df[target]).flatten()

import pandas as pd
full_test_input = pd.concat([train_df[target].tail(lstm_cfg['sequence_length']), test_df[target]])
test_scaled_full = scaler.transform(full_test_input).flatten()

X_train, y_train = create_sequences(train_scaled, lstm_cfg['sequence_length'])
X_test, y_test = create_sequences(test_scaled_full, lstm_cfg['sequence_length'])

lstm = LSTMModel(lstm_cfg['sequence_length'], lstm_cfg['units'], lstm_cfg['dropout'], lstm_cfg['learning_rate'])
lstm.fit(X_train, y_train, batch_size=lstm_cfg['batch_size'], epochs=lstm_cfg['epochs'])
lstm_preds = scaler.inverse_transform(lstm.predict(X_test))
lstm_metrics = compute_metrics(test_df[target].values, lstm_preds)
lstm_metrics

## 3. Train XGBoost + hyperparameter tuning example

In [ ]:
xgb_cfg = model_cfg['xgboost']

def train_and_eval(params):
    model = XGBoostModel(random_state=cfg['seed'], **params)
    model.fit(train_df[feature_cols], train_df[target])
    preds = model.predict(val_df[feature_cols] if len(val_df) else test_df[feature_cols])
    truth = val_df[target] if len(val_df) else test_df[target]
    return compute_metrics(truth.values, preds)['RMSE']

param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [xgb_cfg['subsample']],
    'colsample_bytree': [xgb_cfg['colsample_bytree']],
}

tuning = grid_search(param_grid, train_and_eval, '../experiments/hyperparameter_tuning/tuning_results.csv')
tuning['best_params'], tuning['best_score']

In [ ]:
best_params = tuning['best_params'] or {
    'n_estimators': xgb_cfg['n_estimators'],
    'max_depth': xgb_cfg['max_depth'],
    'learning_rate': xgb_cfg['learning_rate'],
    'subsample': xgb_cfg['subsample'],
    'colsample_bytree': xgb_cfg['colsample_bytree'],
}
xgb = XGBoostModel(random_state=cfg['seed'], **best_params)
xgb.fit(train_df[feature_cols], train_df[target])
xgb_preds = xgb.predict(test_df[feature_cols])
xgb_metrics = compute_metrics(test_df[target].values, xgb_preds)
xgb_metrics

## 4. Log experiments

In [ ]:
log_experiment(paths_cfg['experiment_log'].replace('experiments', '../experiments'), 'exp_arima_nb', 'ARIMA', arima_cfg, arima_metrics)
log_experiment(paths_cfg['experiment_log'].replace('experiments', '../experiments'), 'exp_lstm_nb', 'LSTM', lstm_cfg, lstm_metrics)
log_experiment(paths_cfg['experiment_log'].replace('experiments', '../experiments'), 'exp_xgboost_nb', 'XGBoost', best_params, xgb_metrics)

## 5. Compare models & visualize

In [ ]:
results = {'ARIMA': arima_metrics, 'LSTM': lstm_metrics, 'XGBoost': xgb_metrics}
comparison_df = compare_models(results)
comparison_df.to_csv('../results/tables/model_performance.csv')
comparison_df

In [ ]:
predictions = {'ARIMA': arima_preds, 'LSTM': lstm_preds, 'XGBoost': xgb_preds}
fig = plot_predictions(test_df[target].values, predictions)
save_fig(fig, '../results/plots/comparative_analysis/predictions_overlay.png')

fig2 = plot_metric_comparison(comparison_df)
save_fig(fig2, '../results/plots/comparative_analysis/metric_comparison.png')